# Fase 2 (extra) · Fine-tune QLoRA del Coach

## "Entrenado con datos reales" — la cabecera del proyecto

Adaptamos un modelo abierto pequeno (**Qwen2.5-1.5B-Instruct**) con **QLoRA** (4-bit +
LoRA) sobre datos reales de fitness, para que responda con el formato y tono del coach.
Es la misma tecnica del modulo LLM (M14, "Hablar como Milei").

> **GPU gratis:** Colab T4 (Entorno de ejecucion → GPU). QLoRA en 4-bit cabe de sobra.

> **Nota de arquitectura (honesta):** servir un modelo fine-tuneado 24/7 gratis es
> dificil (los Spaces gratuitos son solo-CPU). Por eso **el coach en produccion usa
> RAG + API gratuita** (notebook 01). Este fine-tune es el **artefacto entrenado** +
> su evaluacion: se sube el adapter al Hub y se demuestra en la presentacion.

---
## 1. Instalacion

> Las APIs de `trl`/`peft` cambian entre versiones. Si `SFTConfig`/`SFTTrainer` diera
> un error de argumentos, ajusta segun la version instalada (se indica abajo).

In [ ]:
!pip -q install -U transformers peft trl bitsandbytes accelerate datasets

import os, torch
from datasets import load_dataset
print("CUDA disponible:", torch.cuda.is_available())

# Token de HF para subir el adapter (Colab secrets -> HF_TOKEN)
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN") or "")
except Exception:
    pass

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_REPO = "TU_USUARIO/forged-coach-qlora"   # <- cambia por tu usuario de HF

---
## 2. Modelo base en 4-bit (QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto")
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM")
print("Modelo base cargado en 4-bit.")

---
## 3. Datos reales → formato chat

Usamos `hammamwahab/fitness-qa` (pregunta → respuesta). Aplicamos el chat template de
Qwen con la misma persona FORGED, para que el modelo aprenda el tono y el formato.

In [ ]:
SYS = ("Eres el Coach de FORGED. Respondes en espanol, directo y sin excusas, "
       "con datos concretos de fitness y nutricion. Cierras con un descargo de que "
       "es informacion de fitness, no consejo medico.")

raw = load_dataset("hammamwahab/fitness-qa", split="train")

def to_text(ex):
    msgs = [
        {"role": "system", "content": SYS},
        {"role": "user", "content": (ex.get("question") or "").strip()},
        {"role": "assistant", "content": (ex.get("answer") or "").strip()},
    ]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

ds = raw.filter(lambda e: (e.get("answer") or "").strip() != "").map(
    to_text, remove_columns=raw.column_names)
print("Ejemplos de entrenamiento:", len(ds))
print(ds[0]["text"][:400])

---
## 4. Entrenamiento (SFT + LoRA)

Pocos pasos: es una demostracion de que el pipeline funciona y el modelo adopta el
estilo. Sube `max_steps` si quieres afinar mas.

In [ ]:
from trl import SFTConfig, SFTTrainer

cfg = SFTConfig(
    output_dir="forged-qlora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=120,
    learning_rate=2e-4,
    logging_steps=10,
    bf16=True,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    dataset_text_field="text",
    max_seq_length=1024,
    report_to="none",
)

trainer = SFTTrainer(model=model, train_dataset=ds, args=cfg, peft_config=lora)
trainer.train()
print("Entrenamiento terminado.")

---
## 5. Prueba: ¿responde como el coach?

In [ ]:
def generar(pregunta, max_new_tokens=220):
    msgs = [{"role": "system", "content": SYS},
            {"role": "user", "content": pregunta}]
    ids = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=max_new_tokens,
                         temperature=0.4, do_sample=True,
                         pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

print(generar("Tengo sobrepeso y quiero empezar. Dame 3 reglas de nutricion."))

---
## 6. Subir el adapter al Hub

Solo se suben los pesos LoRA (unos MB). Este repo es el artefacto "entrenado con datos
reales" que se ensena en la presentacion.

In [ ]:
# Requiere HF_TOKEN con permiso de escritura
if os.environ.get("HF_TOKEN"):
    trainer.model.push_to_hub(OUTPUT_REPO, token=os.environ["HF_TOKEN"])
    tokenizer.push_to_hub(OUTPUT_REPO, token=os.environ["HF_TOKEN"])
    print("Adapter subido a:", OUTPUT_REPO)
else:
    print("Define HF_TOKEN para subir el adapter. Guardado localmente en forged-qlora/.")
    trainer.model.save_pretrained("forged-qlora-adapter")

---
## 7. Conclusiones

- Fine-tune **QLoRA** real sobre datos reales de fitness, en GPU gratuita.
- El modelo adopta el **tono y formato** del coach (evaluable comparando antes/despues).
- Adapter publicado en el Hub como evidencia.

**Que defender:** por que QLoRA (barato, cabe en 4-bit), diferencia con RAG (el RAG
aporta cifras actualizadas y evita alucinaciones; el fine-tune aporta estilo/formato),
y por que en produccion servimos RAG+API (gratis y fiable) reservando el fine-tune como
artefacto entrenado.

### Referencias
- `hammamwahab/fitness-qa` (HF). Qwen2.5 (Alibaba). PEFT/TRL/bitsandbytes (HF).
- Modulo LLM — KeepCoding (M14 fine-tuning / QLoRA).
